In [ ]:
import random
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/all-MiniLM-L6-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "train[:3000]"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 256 if device == "mps" else 64
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})

In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["sentence1"] = df["sentence1"].astype(str)
df["sentence2"] = df["sentence2"].astype(str)
df["label"] = df["label"].astype(np.float32)

print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())

In [ ]:
sentences1 = df["sentence1"].tolist()
sentences2 = df["sentence2"].tolist()
labels = df["label"].to_numpy(dtype=np.float32)

all_sentences = sentences1 + sentences2
sentence_counts = Counter(all_sentences)
unique_sentences = list(dict.fromkeys(all_sentences))

total_encoded_sentences = len(all_sentences)
num_unique_sentences = len(unique_sentences)
duplicate_occurrences = total_encoded_sentences - num_unique_sentences
duplicate_rate = duplicate_occurrences / total_encoded_sentences if total_encoded_sentences else 0.0
num_sentences_repeated = sum(count > 1 for count in sentence_counts.values())
top_duplicate_counts = sentence_counts.most_common(10)

print({
    "total_encoded_sentences": total_encoded_sentences,
    "num_unique_sentences": num_unique_sentences,
    "duplicate_occurrences": duplicate_occurrences,
    "duplicate_rate": round(duplicate_rate, 6),
    "num_sentences_repeated": num_sentences_repeated,
})
print(top_duplicate_counts[:5])

In [ ]:
model = SentenceTransformer(model_name, device=device)
model.eval()

inference_start_time = time.time()

emb1 = model.encode(
    sentences1,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

emb2 = model.encode(
    sentences2,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

inference_seconds = time.time() - inference_start_time

print({
    "embedding_shape_sentence1": tuple(emb1.shape),
    "embedding_shape_sentence2": tuple(emb2.shape),
    "inference_seconds": round(inference_seconds, 2),
})

In [ ]:
cosine_similarity = np.sum(emb1 * emb2, axis=1)
predicted_score_0_5 = 2.5 * (cosine_similarity + 1.0)

results_df = df.copy()
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5

print(results_df[["sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5"]].head(10))

In [ ]:
pearson_corr = pearsonr(predicted_score_0_5, labels).statistic
spearman_corr = spearmanr(predicted_score_0_5, labels).statistic
runtime_seconds = time.time() - start_time

duplicate_summary_df = (
    pd.DataFrame(sentence_counts.items(), columns=["sentence", "count"])
    .sort_values(by=["count", "sentence"], ascending=[False, True])
    .reset_index(drop=True)
)

compact_duplicate_comparison = {
    "total_encoded_sentences_raw": total_encoded_sentences,
    "unique_sentences_if_cached": num_unique_sentences,
    "extra_sentence_encodes_without_cache": duplicate_occurrences,
    "duplicate_rate": round(duplicate_rate, 6),
    "num_sentences_repeated": num_sentences_repeated,
    "max_duplicate_count": int(duplicate_summary_df["count"].max()) if len(duplicate_summary_df) else 0,
}

print({
    "pearson_correlation": round(float(pearson_corr), 6),
    "spearman_correlation": round(float(spearman_corr), 6),
    "runtime_seconds": round(runtime_seconds, 2),
})
print(compact_duplicate_comparison)
print(duplicate_summary_df.head(10).to_dict(orient="records"))

In [ ]:
print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"total_encoded_sentences_raw: {total_encoded_sentences}")
print(f"unique_sentences_if_cached: {num_unique_sentences}")
print(f"extra_sentence_encodes_without_cache: {duplicate_occurrences}")
print(f"duplicate_rate: {duplicate_rate:.6f}")
print(f"num_sentences_repeated: {num_sentences_repeated}")
print(f"pearson_correlation: {pearson_corr:.6f}")
print(f"spearman_correlation: {spearman_corr:.6f}")
print(f"inference_seconds: {inference_seconds:.2f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")
print("top_duplicate_frequency_examples:")
print(duplicate_summary_df.head(10).to_dict(orient="records"))